In [4]:
import requests
import os
import dotenv
from dotenv import load_dotenv

In [5]:
load_dotenv()  # Load environment variables from .env file

False

In [6]:
EDESK_KEY = os.getenv("EDESK_KEY")  


In [7]:
headers = {"Authorization": EDESK_KEY}
url = "https://api.edesk.com/v1/tickets?filter_status_equals=Closed"
response = requests.get(
    url,
    headers=headers
)

In [32]:
print(response.json()["data"][1])

{'id': 680474220, 'subject': 'Questions diverses', 'channel_id': 316593, 'status': 'Closed', 'type': 'OrderQuery', 'sales_order_id': 3177965967, 'sales_order': {'id': 3177965967, 'channel_id': 316593, 'status': 'Delivered', 'seller_order_id': 'F905JA03728-A', 'created_at': '2024-11-23 19:22:22', 'contact_id': 3090461865, 'total_amount': 1619.9, 'shipping_amount': 0, 'tracking_codes': [{'tracking_code': "'1Z62369V6826164776'", 'tracking_link': 'https://wwwapps.ups.com/WebTracking/track?track=yes&trackNums=1Z62369V6826164776', 'tracking_carrier_name': 'UPS'}], 'last_updated_at': '2024-11-28 12:27:32', 'order_items': [{'id': 3320019007, 'quantity': 1, 'end_price': 1619.9, 'product': {'id': 223379627, 'sku': 9000714214, 'title': 'Sedatech PC Gamer Pro Watercooling Full', 'brand': None, 'currency': '€', 'dimensions': None, 'weight': 453.59237, 'product_image_url': 'https://merchant.boulanger.com/media/product/image/0fbc356d-0304-436b-b5d9-64391ed39b86', 'price': 1619.9, 'marketplace_link': 

In [119]:
import sqlite3, json
conn = sqlite3.connect("tickets.db")

no_customer = 0
no_agent = 0
empty_after_join = 0

tickets = conn.execute("SELECT ticket_id FROM tickets").fetchall()
for (ticket_id,) in tickets:
    messages = conn.execute(
        "SELECT sender_role, body FROM messages WHERE ticket_id = ?", (ticket_id,)
    ).fetchall()
    customer_msgs = [m for m in messages if m[0] == "Customer"]
    agent_msgs = [m for m in messages if m[0] == "Agent"]
    if not customer_msgs:
        no_customer += 1
    elif not agent_msgs:
        no_agent += 1
    elif not "\n".join(b for r, b in customer_msgs if b).strip():
        empty_after_join += 1

print("no customer messages:", no_customer)
print("no agent messages:", no_agent)
print("empty after join:", empty_after_join)

no customer messages: 2631
no agent messages: 2143
empty after join: 6


In [121]:

import sqlite3
cnx = sqlite3.connect('tickets.db')
cursor = cnx.cursor()
cursor.execute("SELECT DISTINCT sender_role FROM messages;")
print(cursor.fetchall())
cnx.close()

[('Customer',), ('Agent',)]


In [17]:

import sqlite3
cnx = sqlite3.connect('chunks.db')
cursor = cnx.cursor()
cursor.execute("UPDATE chunks SET embedded = 0 WHERE embedded = 1;")
cnx.commit()
print(cursor.fetchall())
cnx.close()

[]


In [29]:
cnx = sqlite3.connect('tickets.db')
cursor = cnx.cursor()
cursor.execute("PRAGMA table_info(messages);")
print(cursor.fetchall())
cnx.close()

[(0, 'message_id', 'INTEGER', 0, None, 1), (1, 'ticket_id', 'INTEGER', 1, None, 0), (2, 'sender_role', 'TEXT', 0, None, 0), (3, 'body', 'TEXT', 0, None, 0), (4, 'created_at', 'TEXT', 0, None, 0), (5, 'is_solution', 'INTEGER', 0, '0', 0), (6, 'language', 'TEXT', 0, None, 0)]


In [33]:
cnx = sqlite3.connect('tickets.db')
cursor = cnx.cursor()
cursor.execute("SELECT body FROM messages WHERE ticket_id = 677864340;")
print(cursor.fetchall())
cnx.close()

[('Objet: Demande de partenariat / soutien pour mes streams\n\nBonjour,\n\nJe me permets de vous contacter concernant ce PC que vous vendez.\n\nJe suis créateur de contenu et je fais des streams en ligne, mais malheureusement mon PC actuel est tombé en panne récemment. Cela m’empêche de continuer mes streams, et à l’heure actuelle je n’ai pas les moyens financiers nécessaires pour en acheter un nouveau.\n\nJe souhaitais donc savoir s’il vous serait éventuellement possible de m’offrir ce PC, ou de me proposer une aide exceptionnelle. En échange, je m’engage à mentionner votre boutique lors de mes streams, à vous taguer sur mes réseaux et à faire de la publicité pour vos produits auprès de ma communauté.\n\nJe comprends parfaitement que ce type de demande soit particulier, et je vous remercie sincèrement d’avoir pris le temps de lire mon message, quelle que soit votre réponse.\n\nJe reste à votre disposition pour toute information complémentaire.\n\nCordialement,\nPiveteau Ethan / ethanp

In [ ]:
headers = {"Authorization": EDESK_KEY}
url = "https://api.edesk.com/v1/tickets/680474220?include=messages"
response = requests.get(
    url,
    headers=headers
)


In [100]:
import sqlite3
conn = sqlite3.connect("tickets.db")

# How many tickets have zero messages vs at least one?
print("tickets with messages:", conn.execute("""
    SELECT COUNT(DISTINCT ticket_id) FROM messages
""").fetchone())

print("total tickets:", conn.execute("SELECT COUNT(*) FROM tickets").fetchone())

# Which tickets are missing messages entirely?
missing = conn.execute("""
    SELECT t.ticket_id FROM tickets t
    LEFT JOIN messages m ON t.ticket_id = m.ticket_id
    WHERE m.ticket_id IS NULL
    LIMIT 10
""").fetchall()
print("sample tickets with no messages:", missing)

tickets with messages: (8041,)
total tickets: (10000,)
sample tickets with no messages: [(496665059,), (497363785,), (498325972,), (498385835,), (498625451,), (498928390,), (500593989,), (501434122,), (501622197,), (501912072,)]


In [43]:
response.json()["data"]["messages_ids"]

[3415493292,
 3415493296,
 3415493310,
 3417183677,
 3417229997,
 3417230016,
 3563209849,
 3563209851,
 3563792340,
 3563937977,
 3563937983]

In [8]:
headers = {"Authorization": EDESK_KEY}
url = "https://api.edesk.com/v1/messages/3563937983"
response = requests.get(
    url,
    headers=headers
)
response.json()["data"]

KeyError: 'data'

In [10]:
import json
tickets = json.load(open("data/tickets.json"))

In [11]:
tickets[1111]["messages"][0]

{'is_incoming': '1',
 'created_at': '2025-08-11 13:53:11',
 'message_body': 'Thank you for delivering the computer for order No. 305-2577694-0095527 via Amazon.de. We are very satisfied with the build quality and configuration, and it would be a great pity if we had to return the product due to an invoicing error.',
 'type': 'Consumer Conversation',
 'language': 'English (UK)',
 'delivery_status': 'pending delivery',
 'response_time': 0,
 'handling_time': 0}

In [34]:
import pyodbc

server = "W2012R2-SQL12BI"      # or hostname, e.g. "DESKTOP-ABC123"
database = """ShopCenter2014"""
username = "W2012R2-SQL12BI"
password = "Q47RgEIffYevQR"

conn = pyodbc.connect(
    f"DRIVER={{ODBC Driver 18 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={username};"
    f"PWD={password};"
    "TrustServerCertificate=yes;"
)

cursor = conn.cursor()

cursor.execute("SELECT @@VERSION")
print(cursor.fetchone()[0])

conn.close()

ImportError: libodbc.so.2: cannot open shared object file: No such file or directory